# 第1章 ダイオード／LED の I-V カーブトレーサ（ノートブック版）

教科書の第1章のうち、**Pico がなくても動く解析の部分**（1-6 の `iv_analysis.py`・`check_resistor.py`・`planck.py`）を、合成の I-V データで試せるように1冊にまとめたものです。上から順に実行してください（Mac の Jupyter でも Google Colab でも動きます。執筆時の Mac で全部で約2分。PyMC のコンパイルとサンプリングがあるので、Colab ではその数倍かかることがあります）。

| 節 | 試すこと | 教科書の手順 | 使うファイル |
|---|---|---|---|
| 1 | 合成の I-V データを作る（Pico A の測定の代わり） | 手順 6〜9 の測定の代わり | `synth_iv.py`（ノートブック版で追加） |
| 2 | 抵抗の校正（傾き = 抵抗値） | 手順 6 | `check_resistor.py` |
| 3 | 重みつき最小二乗（Lambert W の自己診断つき） | 手順 10（`--no-bayes`） | `iv_analysis.py` |
| 4 | PyMC でのベイズ推定と、事後サンプルの保存（**ノートブックでは小さい設定**） | 手順 10、1-3-4 | `iv_analysis.py` |
| 5 | 識別可能性：$R_s$ が決まるとき・決まらないとき | 1-3-3・1-3-4 のコツ、1-7 | `iv_analysis.py` |
| 6 | LED 3色からプランク定数 | 手順 11、1-3-5 | `planck.py` |

- 次のセルたちが、1-6 の Mac 側の解析のファイル3つと、合成データを作る `synth_iv.py` を、このノートブックの隣に `%%writefile` で書き出します（教科書の `~/home-lab/ch1/` の代わり）。中身は 1-6 と同じです。変えたのは、図の日本語フォントの候補を足したこと（`plt.rcParams["font.family"]` の行）と、`iv_analysis.py` に `--tune`（NUTS の調整の回数。既定は本文と同じ 1500）を足したことだけです。
- `synth_iv.py` は本文にはありません。図1-7 の回路（PWM → R_F → DUT → R_L、H レンジでは 2N7000＋R_H が並列）の定常状態を、1-3-2 の式（$V = nV_T\ln(1+I/I_s) + IR_s$）で解き、1-3-3 の雑音（1回の読みで約 1.9 mV、64 回平均）と ADC のオフセットを足して、`iv_capture.py` と同じ形の CSV と JSON を `data_iv/` に作ります。真の値は `○○_synth_truth.json` に書き、ノートブックの確かめだけに使います（解析のプログラムには知らせません）。ADC の DNL のような系統誤差は入れていません。
- 各節では、ターミナルの `uv run python ○○.py …` の代わりに、`run("○○.py …")` で同じ引数のまま実行します（中では `python ○○.py …` を別のプロセスで動かします）。表示を見てから、`ok()` で教科書の手順の「確かめること」を、合成の真の値とも比べながら確かめます。
- **4・5 節のベイズ推定は重い**ので、ノートブックでは小さい設定（`DRAWS = 300`、`TUNE = 300`、4 チェーン）を既定にしています。教科書の本番（`--draws 1000`、tune 1500）にするときは、4 節のセルの `DRAWS`・`TUNE` を書きかえてください。ノートブックの確かめは、設定に関係なく成り立つこと（真の値が 95%HDI に入る、R-hat、事後サンプルのファイルの形など）にしてあります。R-hat は、本文の目安は 1.01 未満ですが、小さい設定では少し超えることがあるので、`ok()` は 1.05 未満で確かめ、1.01 未満かどうかは表示だけにしています。
- **入れていないもの（Pico A が要る部分）**：回路を組むこと、`iv_tracer.py` の書き込みと `PING`・`PADS`・`SET`・`READ` の確認（手順 2〜5）、`iv_capture.py` での実物の測定（手順 6〜9）。この2つのプログラムは、下に読むためだけに載せてあります（実行しません）。自分の測定データ（`data_iv/○○.csv` と `.json`）を Colab の左のファイルの欄に上げれば、同じ `run("iv_analysis.py data_iv/○○.csv")` で解析できるはずですが、執筆時には確かめていません。
- ⚡ 実物の測定の注意（C_F の極性、LED に直接 3.3 V をかけないことなど）は、教科書の 1-4・1-5 を読んでください。このノートブックは計算だけです。
- 乱数の種を変えなければ数値はほぼ同じですが、PyMC・PyTensor・numpy の版で最後の桁が変わることがあります。
- Colab では、最初のほうの「0-1. 図の日本語のフォント」のセルが日本語のフォント（Noto Sans CJK JP）を入れます（数十秒〜1分ほど。Mac では Hiragino Sans を使うので何もしません）。そのセルが「★日本語のフォントが見つからない」と表示したときは、図の日本語が四角（□）になります（数値の確認には影響しません）。

In [ ]:
# 0. 準備：足りないパッケージだけ入れる（Colab には numpy・scipy・pandas・matplotlib・statsmodels と、たいてい PyMC も入っている。版は要確認）
import importlib, subprocess, sys
for mod, pkg in [("pymc", "pymc"), ("statsmodels", "statsmodels"), ("pandas", "pandas"), ("scipy", "scipy")]:
    try:
        importlib.import_module(mod)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
import pymc, pytensor, statsmodels, scipy, numpy, pandas
print("OK", "PyMC", pymc.__version__, "PyTensor", pytensor.__version__, "statsmodels", statsmodels.__version__,
      "scipy", scipy.__version__, "numpy", numpy.__version__, "pandas", pandas.__version__)
print("（ノートブックは PyMC 6.4.0・PyTensor 3.3.8・statsmodels 0.15.0 で確かめた）")

In [ ]:
# 0-1. 準備：図の日本語のフォント（Colab のときだけ fonts-noto-cjk を入れる。Mac は Hiragino Sans を使うので何もしない）
import glob, os, subprocess, sys
import matplotlib
from matplotlib import font_manager

JP_FONTS = ["Hiragino Sans", "Noto Sans CJK JP", "IPAexGothic"]   # 本文の図と同じ候補（上から順に使う）


def find_jp_font():
    have = {f.name for f in font_manager.fontManager.ttflist}
    return next((n for n in JP_FONTS if n in have), None)


IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and find_jp_font() is None:
    print("Colab：日本語のフォント（fonts-noto-cjk）を入れます（数十秒〜1分ほど）")
    cmd = ["apt-get", "install", "-y", "-qq", "fonts-noto-cjk"]
    if subprocess.run(cmd, capture_output=True).returncode != 0:
        subprocess.run(["apt-get", "update", "-qq"], capture_output=True)   # パッケージの一覧が古いときは、更新してもう一度
        subprocess.run(cmd, capture_output=True)
    for path in glob.glob("/usr/share/fonts/opentype/noto/NotoSansCJK*.ttc"):
        font_manager.fontManager.addfont(path)   # このノートブックの中でかく図に効く（再起動は要らない）
    # 別のプロセスで動かすスクリプト（subprocess や !python）にも見えるように、フォントの一覧のキャッシュを書き直す
    font_manager.json_dump(font_manager.fontManager, os.path.join(
        matplotlib.get_cachedir(), f"fontlist-v{font_manager.FontManager.__version__}.json"))

JP_FONT = find_jp_font()
if JP_FONT:
    matplotlib.rcParams["font.family"] = [JP_FONT, "sans-serif"]
    matplotlib.rcParams["font.sans-serif"] = [JP_FONT] + [f for f in matplotlib.rcParams["font.sans-serif"] if f != JP_FONT]
    if IN_COLAB:   # 本文のスクリプトの "sans-serif" も日本語のフォントになるように、Colab の matplotlib の設定ファイルに書く
        rc = os.path.join(matplotlib.get_configdir(), "matplotlibrc")
        lines = open(rc, encoding="utf-8").read().splitlines() if os.path.exists(rc) else []
        lines = [l for l in lines if not l.startswith("font.sans-serif")]
        lines.append("font.sans-serif: " + ", ".join(matplotlib.rcParams["font.sans-serif"]))
        with open(rc, "w", encoding="utf-8") as f:
            f.write("\n".join(lines) + "\n")
    print("図の日本語のフォント：", JP_FONT)
else:
    print("★日本語のフォントが見つからないので、図の日本語が四角（□）になります（数値には影響しません）。")
    print("  Colab なら、インターネットにつながっているかを確かめて、このセルをもう一度実行してください。"
          "Linux の Jupyter なら、sudo apt install fonts-noto-cjk で入れてから、カーネルを再起動します。")

## Pico A の側（読むだけ。実行しません）

教科書の手順では、Pico A に `iv_tracer.py`（MicroPython。`main.py` として保存）を書き込み、Mac の `iv_capture.py` がシリアルで `SWEEP L|H npts settle_ms nsamp updown` を送って、L レンジと H レンジを往復で掃引します。各点では、デューティを変えてから 600 ms（時定数 $R_FC_F = 0.1$ s の約6倍）待ち、ADC0〜2（節点 A・K・S）を 64 回ずつ読んだ平均と標準偏差を `DATA,…` の行で返します。`iv_capture.py` はそれを `data_iv/○○_日時.csv`（列 `dut,rep,rng,dir,idx,duty,vA,vK,vS,sdA,sdK,sdS`）と、条件を書いた同じ名前の `.json`（`temp_C`・`vref`・`R_L`・`R_H`・`nsamp`・`wavelength_nm` など）に保存します。このノートブックの 1 節の `synth_iv.py` は、この2つのファイルと同じ形のものを作ります。

1-6 の2つのプログラムを、そのまま載せておきます（Colab では Pico につなげないので、実行しません）。

**Pico A（MicroPython）｜~/home-lab/ch1/iv_tracer.py（Pico には main.py として保存）**（読むだけ）

```python
# 第1章 I-Vカーブトレーサ（Pico 2 WH / MicroPython）
# 書き込み: mpremote connect auto fs cp iv_tracer.py :main.py
#           mpremote connect auto reset
#
# 配線（図1-7）
#   GP15(ピン20) → R_F 1kΩ → 節点A（C_F 100µF の＋）→ DUTアノード
#   DUTカソード = 節点K → R_L 10kΩ → AGND、節点K → 2N7000 ドレイン
#   2N7000 ソース = 節点S → R_H 100Ω → AGND
#   GP16(ピン21) → 100Ω → ゲート、ゲート → 8.2kΩ → GND
#   GP26(ピン31)=節点A、GP27(ピン32)=節点K、GP28(ピン34)=節点S、AGND(ピン33)
#
# シリアルコマンド（1行ずつ）
#   PING                       → PONG
#   VREF 3.296                 → ADC基準電圧[V]（ADC_VREFピンの実測値）を設定
#   PADS                       → ADCピンのパッド設定を表示（E9対策の確認）
#   SET 32768                  → PWMデューティ（0〜65535）
#   RANGE 0|1                  → 0: Lレンジ（FETオフ）、1: Hレンジ（FETオン）
#   READ 64                    → 3ch を64回平均して表示
#   SWEEP L|H npts settle_ms nsamp up|down|updown
#                              → 掃引して DATA 行を出力（最後に END）
import sys
import time
from machine import Pin, PWM, ADC, mem32

PWM_PIN = 15
RANGE_PIN = 16
ADC_GPIOS = (26, 27, 28)      # 節点A, 節点K, 節点S
PWM_FREQ = 100_000            # 100 kHz（RCフィルタ後のリプルを十分小さくする）
vref = 3.300                  # VREF コマンドで上書きする

# ---- RP2350 エラッタ E9 対策：ADC ピンのデジタル入力バッファとプル抵抗を切る ----
# PADS_BANK0 のベースアドレスとビット位置は RP2350 データシート「PADS_BANK0」の章で要確認。
PADS_BANK0_BASE = 0x40038000
IE_BIT = 1 << 6      # Input Enable
PUE_BIT = 1 << 3     # Pull-up Enable
PDE_BIT = 1 << 2     # Pull-down Enable


def pad_addr(gpio):
    return PADS_BANK0_BASE + 4 + 4 * gpio     # 先頭4バイトは VOLTAGE_SELECT


def pad_value(gpio):
    return mem32[pad_addr(gpio)] & 0xFFFFFFFF


def disable_digital_input(gpio):
    v = pad_value(gpio)
    if v & ~0x1FF:                             # パッドレジスタは下位9ビットのみ。違えば番地違い
        print("WARN pad register looks wrong: GP%d = 0x%08x（書き換えません）" % (gpio, v))
        return
    mem32[pad_addr(gpio)] = v & ~(IE_BIT | PUE_BIT | PDE_BIT)


pwm = PWM(Pin(PWM_PIN))
pwm.freq(PWM_FREQ)
pwm.duty_u16(0)
rng_pin = Pin(RANGE_PIN, Pin.OUT, value=0)     # 起動時は L レンジ（外付けプルダウンでも保証）
adcs = [ADC(Pin(g)) for g in ADC_GPIOS]
for g in ADC_GPIOS:                            # ADC を初期化した「あと」に入力バッファを切る
    disable_digital_input(g)


def safe_state():
    pwm.duty_u16(0)
    rng_pin.value(0)


def read_all(nsamp):
    """3ch を交互に nsamp 回ずつ読み、平均[V]と標準偏差[V]（1回ごとの読みのばらつき）を返す"""
    s = [0, 0, 0]
    s2 = [0, 0, 0]
    for _ in range(nsamp):
        for k in range(3):
            x = adcs[k].read_u16() >> 4        # 16ビット表現 → 12ビット値（0〜4095）
            s[k] += x
            s2[k] += x * x
    mean_v, sd_v = [], []
    for k in range(3):
        m = s[k] / nsamp
        var = s2[k] / nsamp - m * m
        if var < 0:
            var = 0.0
        mean_v.append(m * vref / 4096)
        sd_v.append((var ** 0.5) * vref / 4096)
    return mean_v, sd_v


def sweep(rng, npts, settle_ms, nsamp, mode):
    if npts < 2:
        raise ValueError("npts は 2 以上")
    rng_pin.value(1 if rng == "H" else 0)
    up = [round(65535 * i / (npts - 1)) for i in range(npts)]
    if mode == "up":
        duties = up
    elif mode == "down":
        duties = up[::-1]
    else:                                       # updown：往復（折り返し点は1回だけ）
        duties = up + up[::-1][1:]
    pwm.duty_u16(0)
    time.sleep_ms(1500)                         # C_F を放電させてから開始
    print("BEGIN,%s,%d,%d,%d,%s,%.4f" % (rng, npts, settle_ms, nsamp, mode, vref))
    for i, d in enumerate(duties):
        pwm.duty_u16(d)
        time.sleep_ms(settle_ms)                # 時定数 0.1 s の約6倍待つ
        m, sd = read_all(nsamp)
        print("DATA,%d,%d,%.5f,%.5f,%.5f,%.5f,%.5f,%.5f"
              % (i, d, m[0], m[1], m[2], sd[0], sd[1], sd[2]))
    safe_state()
    print("END")


def handle(line):
    global vref
    p = line.strip().split()
    if not p:
        return
    c = p[0].upper()
    if c == "PING":
        print("PONG")
    elif c == "VREF":
        v = float(p[1])
        if not 3.0 < v < 3.6:
            raise ValueError("VREF が範囲外")
        vref = v
        print("OK")
    elif c == "PADS":
        for g in ADC_GPIOS:
            v = pad_value(g)
            print("PAD,GP%d,0x%03x,IE=%d,PUE=%d,PDE=%d"
                  % (g, v, (v >> 6) & 1, (v >> 3) & 1, (v >> 2) & 1))
        print("OK")
    elif c == "SET":
        d = max(0, min(65535, int(p[1])))
        pwm.duty_u16(d)
        print("OK")
    elif c == "RANGE":
        rng_pin.value(1 if p[1] == "1" else 0)
        print("OK")
    elif c == "READ":
        n = int(p[1]) if len(p) > 1 else 64
        m, sd = read_all(n)
        print("VALS,%.5f,%.5f,%.5f,%.5f,%.5f,%.5f" % (m[0], m[1], m[2], sd[0], sd[1], sd[2]))
        print("OK")
    elif c == "SWEEP":
        rng = p[1].upper()
        if rng not in ("L", "H"):
            raise ValueError("レンジは L か H")
        sweep(rng, int(p[2]), int(p[3]), int(p[4]), p[5].lower() if len(p) > 5 else "up")
    else:
        raise ValueError("unknown command: " + c)


def main():
    print("READY iv_tracer")
    while True:
        line = sys.stdin.readline()
        try:
            handle(line)
        except Exception as e:
            safe_state()
            print("ERR", e)


try:
    main()
finally:                                        # Ctrl-C で止めたときも安全な状態に戻す
    safe_state()
```

**Mac｜~/home-lab/ch1/iv_capture.py**（読むだけ）

```python
#!/usr/bin/env python3
"""第1章：Pico に掃引コマンドを送り、結果を CSV（データ）＋JSON（条件）に保存する
使い方の例:
  uv run python iv_capture.py --dut 1N4148 --temp 22.5 --vref 3.296 --rl 9980 --rh 99.7 --reps 3
  uv run python iv_capture.py --dut LED_RED --wavelength 625 --temp 22.5 --vref 3.296 --rl 9980 --rh 99.7
"""
import argparse
import csv
import datetime
import glob
import json
import pathlib
import sys
import time

import serial


def find_port():
    ports = sorted(glob.glob("/dev/cu.usbmodem*"))
    if not ports:
        sys.exit("Pico のシリアルポートが見つかりません（データ通信対応ケーブルか確認）")
    if len(ports) > 1:
        print("複数のポートがあります。--port で指定してください:", ports)
    return ports[0]


class Pico:
    def __init__(self, port):
        self.s = serial.Serial(port, 115200, timeout=1)
        time.sleep(0.3)
        self.s.reset_input_buffer()

    def send(self, line):
        self.s.write((line + "\n").encode())

    def readline(self):
        return self.s.readline().decode(errors="ignore").strip()

    def ping(self, tries=10):
        for _ in range(tries):
            self.send("PING")
            t0 = time.time()
            while time.time() - t0 < 1.0:
                if self.readline() == "PONG":
                    return True
        return False

    def command(self, line, end_tokens=("OK", "END"), timeout=1200, progress=False):
        self.send(line)
        out = []
        t0 = time.time()
        while time.time() - t0 < timeout:
            ln = self.readline()
            if not ln:
                continue
            if ln.startswith("ERR"):
                raise RuntimeError(ln)
            out.append(ln)
            if progress and ln.startswith("DATA,"):
                idx = int(ln.split(",")[1])
                if idx % 20 == 0:
                    print(f"   点 {idx}", flush=True)
            if ln in end_tokens:
                return out
        raise TimeoutError(line)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--dut", required=True, help="素子名（ファイル名に使う。例 1N4148, LED_RED, R1k）")
    ap.add_argument("--wavelength", type=float, default=None, help="LEDの公称ピーク波長 [nm]")
    ap.add_argument("--temp", type=float, required=True, help="室温 [°C]")
    ap.add_argument("--vref", type=float, required=True, help="ADC_VREF の実測値 [V]")
    ap.add_argument("--rl", type=float, required=True, help="R_L の実測値 [Ω]")
    ap.add_argument("--rh", type=float, required=True, help="R_H の実測値 [Ω]")
    ap.add_argument("--rf", type=float, default=1000.0, help="R_F の値 [Ω]（記録用）")
    ap.add_argument("--npts", type=int, default=121)
    ap.add_argument("--settle", type=int, default=600, help="各点の待ち時間 [ms]")
    ap.add_argument("--nsamp", type=int, default=64, help="各点の平均回数")
    ap.add_argument("--reps", type=int, default=1, help="繰り返し回数")
    ap.add_argument("--port", default=None)
    ap.add_argument("--outdir", default="data_iv")
    a = ap.parse_args()

    p = Pico(a.port or find_port())
    if not p.ping():
        sys.exit("Pico から PONG が返りません。main.py が動いているか確認（mpremote connect auto reset）")
    p.command(f"VREF {a.vref:.4f}")

    rows = []
    for rep in range(a.reps):
        for rng in ("L", "H"):
            print(f"rep {rep} range {rng} 掃引中（約 {2*a.npts*(a.settle+10)/1000/60:.1f} 分）")
            lines = p.command(f"SWEEP {rng} {a.npts} {a.settle} {a.nsamp} updown", progress=True)
            for ln in lines:
                if not ln.startswith("DATA,"):
                    continue
                f = ln.split(",")
                idx = int(f[1])
                rows.append({"dut": a.dut, "rep": rep, "rng": rng,
                             "dir": "up" if idx < a.npts else "down", "idx": idx,
                             "duty": int(f[2]),
                             "vA": float(f[3]), "vK": float(f[4]), "vS": float(f[5]),
                             "sdA": float(f[6]), "sdK": float(f[7]), "sdS": float(f[8])})

    outdir = pathlib.Path(a.outdir)
    outdir.mkdir(parents=True, exist_ok=True)
    stamp = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
    base = outdir / f"{a.dut}_{stamp}"
    with open(base.with_suffix(".csv"), "w", newline="") as fp:
        w = csv.DictWriter(fp, fieldnames=list(rows[0].keys()))
        w.writeheader()
        w.writerows(rows)
    meta = {"dut": a.dut, "wavelength_nm": a.wavelength, "temp_C": a.temp, "vref": a.vref,
            "R_L": a.rl, "R_H": a.rh, "R_F": a.rf, "npts": a.npts, "settle_ms": a.settle,
            "nsamp": a.nsamp, "reps": a.reps, "datetime": stamp}
    base.with_suffix(".json").write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"保存しました: {base}.csv / .json（{len(rows)} 点）")


if __name__ == "__main__":
    main()
```

次の4つのセルで、解析のファイルを書き出します（`check_resistor.py`・`planck.py` は、同じフォルダの `iv_analysis.py` の `load` を読み込みます）。最後の `synth_iv.py` は、ノートブック版で追加した合成データのプログラムです。

In [ ]:
%%writefile iv_analysis.py
#!/usr/bin/env python3
"""第1章 I-V データ解析
- 測定 CSV から V, I を計算
- ショックレー式＋直列抵抗（Lambert W で陽に解いた形）を重みつき最小二乗で当てはめ
- 同じモデルを PyMC でベイズ推定（n, Is, Rs の事後分布と 95% HDI）
- 事後サンプル（log10_Is・n・Rs の組と室温）を *_samples.npz に保存（第6章で回路モデルに通す）
使い方:
  uv run python iv_analysis.py data_iv/1N4148_20261121-101500.csv
  uv run python iv_analysis.py data_iv/1N4148_20261121-101500.csv --no-bayes
"""
import argparse
import json
import pathlib

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")                  # 画面に出さず PNG に保存する
import matplotlib.pyplot as plt
from scipy.optimize import least_squares
from scipy.special import lambertw

plt.rcParams["font.family"] = ["Hiragino Sans", "Noto Sans CJK JP", "IPAexGothic", "sans-serif"]   # ノートブック版：Colab でも止まらないように候補を足す
plt.rcParams["axes.unicode_minus"] = False

Q = 1.602176634e-19                    # 電気素量 [C]（SI 定義値）
KB = 1.380649e-23                      # ボルツマン定数 [J/K]（SI 定義値）
LN10 = np.log(10.0)


# ------------------------------------------------------------------ データ読み込み
def load(csv_path):
    """CSV と同名の JSON を読み、V・I・電流の標準誤差を計算する"""
    csv_path = pathlib.Path(csv_path)
    df = pd.read_csv(csv_path)
    meta = json.loads(csv_path.with_suffix(".json").read_text(encoding="utf-8"))
    RL, RH = meta["R_L"], meta["R_H"]
    df["r"] = (df["rng"] == "H").astype(int)            # 0: L レンジ, 1: H レンジ
    df["V"] = df["vA"] - df["vK"]                       # DUT 両端の電圧 [V]
    # DUT を流れる電流 [A]。R_H の経路の分は H レンジ（FET オン）のときだけ足す。
    # L レンジでは FET が切れていて vS は ADC のオフセットと雑音だけなので、vS/R_H を足すと約 2µA の雑音が入る（1-3-3）
    df["I"] = df["vK"] / RL + df["r"] * df["vS"] / RH
    # 平均値の標準誤差 = 1回の読みの標準偏差 / sqrt(平均回数)
    df["sI"] = np.sqrt((df["sdK"] / RL) ** 2 + df["r"] * (df["sdS"] / RH) ** 2) / np.sqrt(meta["nsamp"])
    return df, meta


# ------------------------------------------------------------------ モデル
def lambertw_exp_np(x, iters=8):
    """w = W(exp(x)) を w + ln w = x のニュートン法で解く（exp(x) を直接計算しない）"""
    x = np.maximum(x, -700.0)
    w = np.where(x > 1.0, x - np.log(np.maximum(x, 1.0)), np.exp(np.minimum(x, 1.0)))
    for _ in range(iters):
        w = w * (1.0 + x - np.log(w)) / (1.0 + w)
    return w


def diode_current(V, log10Is, n, logRs, VT):
    """直列抵抗つきショックレー式の電流 [A]（Lambert W 形）"""
    a = n * VT
    Is = 10.0 ** log10Is
    Rs = np.exp(logRs)
    x = log10Is * LN10 + logRs - np.log(a) + (V + Is * Rs) / a
    return a / Rs * lambertw_exp_np(x) - Is


def model_I(theta, V, r, VT):
    """theta = [log10 Is, n, ln Rs, オフセットL[µA], オフセットH[µA]]"""
    log10Is, n, logRs, offL, offH = theta
    off = np.where(r == 1, offH, offL) * 1e-6
    return diode_current(V, log10Is, n, logRs, VT) + off


def check_lambertw():
    x = np.linspace(-30, 60, 200)
    ref = lambertw(np.exp(x)).real
    err = np.max(np.abs(lambertw_exp_np(x) - ref) / ref)
    print(f"[自己診断] 自作 Lambert W と scipy の最大相対誤差: {err:.1e}")


# ------------------------------------------------------------------ 最小二乗
def sigma_total(d, meta):
    """各点の電流の標準偏差の見積もり：平均の標準誤差＋量子化（1LSB/√12）＋相対1%"""
    lsb = meta["vref"] / 4096
    q = np.where(d["r"] == 1, lsb / meta["R_H"], lsb / meta["R_L"]) / np.sqrt(12)
    return np.sqrt(d["sI"].to_numpy() ** 2 + q ** 2 + (0.01 * d["I"].to_numpy()) ** 2)


def initial_guess(d, VT):
    """L レンジの 1〜50 µA の点で ln I を V に直線回帰して n と Is の初期値を作る"""
    s = d[(d["r"] == 0) & (d["I"] > 1e-6) & (d["I"] < 50e-6)]
    if len(s) >= 5:
        b, a = np.polyfit(s["V"], np.log(s["I"]), 1)    # ln I = a + b V
        n0, log10Is0 = 1.0 / (b * VT), a / LN10
    else:
        n0, log10Is0 = 2.0, -12.0
    return np.array([log10Is0, float(np.clip(n0, 0.9, 5.5)), np.log(5.0), 0.0, 0.0])


def fit_lsq(d, VT, theta0, meta):
    V, I, r = d["V"].to_numpy(), d["I"].to_numpy(), d["r"].to_numpy()
    s = sigma_total(d, meta)
    lb = np.array([-45.0, 0.8, np.log(1e-3), -50.0, -500.0])
    ub = np.array([-2.0, 6.0, np.log(1e4), 50.0, 500.0])
    theta0 = np.clip(theta0, lb + 1e-6, ub - 1e-6)
    res = least_squares(lambda th: (model_I(th, V, r, VT) - I) / s, theta0,
                        bounds=(lb, ub), x_scale="jac")
    dof = max(len(I) - len(theta0), 1)
    chi2_red = 2 * res.cost / dof                       # 規格化カイ二乗（1に近いのが理想）
    cov = np.linalg.pinv(res.jac.T @ res.jac) * chi2_red  # ヤコビアンからの近似共分散
    return res.x, cov, chi2_red


# ------------------------------------------------------------------ ベイズ推定
def lambertw_exp_pt(x, iters=8):
    """PyTensor 版。微分可能なので NUTS の中でそのまま使える"""
    import pytensor.tensor as pt
    x = pt.maximum(x, -700.0)
    w = pt.switch(x > 1.0, x - pt.log(pt.maximum(x, 1.0)), pt.exp(pt.minimum(x, 1.0)))
    for _ in range(iters):
        w = w * (1.0 + x - pt.log(w)) / (1.0 + w)
    return w


def fit_bayes(d, VT, theta0, draws, seed=20261121, tune=1500):   # ノートブック版：tune を引数に
    import pymc as pm
    import pytensor.tensor as pt
    V = d["V"].to_numpy()
    I_uA = d["I"].to_numpy() * 1e6                      # 尤度は µA 単位で書く（数値が扱いやすい）
    r = d["r"].to_numpy()
    with pm.Model():
        log10_Is = pm.Normal("log10_Is", mu=-14.0, sigma=6.0)
        n = pm.TruncatedNormal("n", mu=2.0, sigma=1.0, lower=0.8, upper=6.0)
        log_Rs = pm.Normal("log_Rs", mu=np.log(5.0), sigma=2.0)
        pm.Deterministic("Rs", pt.exp(log_Rs))
        z_off = pm.Normal("z_off", 0.0, 1.0, shape=2)  # 非中心化したオフセット
        off_uA = pm.Deterministic("off_uA", z_off * np.array([1.0, 100.0]))
        log_sa = pm.Normal("log_sigma_a", mu=np.log(np.array([0.1, 10.0])), sigma=1.5, shape=2)
        sigma_rel = pm.HalfNormal("sigma_rel", sigma=0.05)

        a = n * VT
        Is = 10.0 ** log10_Is
        Rs = pt.exp(log_Rs)
        x = log10_Is * LN10 + log_Rs - pt.log(a) + (V + Is * Rs) / a
        mu = (a / Rs * lambertw_exp_pt(x) - Is) * 1e6 + off_uA[r]
        sig = pt.sqrt(pt.exp(log_sa)[r] ** 2 + (sigma_rel * mu) ** 2)
        pm.Normal("I_obs", mu=mu, sigma=sig, observed=I_uA)

        init = {"log10_Is": float(theta0[0]), "n": float(np.clip(theta0[1], 0.85, 5.9)),
                "log_Rs": float(theta0[2]),
                "z_off": np.array([theta0[3] / 1.0, theta0[4] / 100.0])}
        idata = pm.sample(draws=draws, tune=tune, chains=4, cores=4, target_accept=0.95,
                          initvals=init, random_seed=seed)
    return idata


# ------------------------------------------------------------------ 事後分布の要約（ArviZ の版に依存しないように numpy で計算）
def rhat(x):
    """分割した R-hat（Gelman–Rubin）。x は (チェーン数, 1チェーンの点数)。1.01 未満が目安"""
    h = x.shape[1] // 2
    x = np.concatenate([x[:, :h], x[:, h:2 * h]])       # 各チェーンを前半・後半に分ける
    n = x.shape[1]
    w = x.var(axis=1, ddof=1).mean()
    b = n * x.mean(axis=1).var(ddof=1)
    return float(np.sqrt(((n - 1) / n * w + b / n) / w))


def hdi(x, prob=0.95):
    """最高事後密度区間（HDI）：サンプルの prob を含む区間のうち、いちばん短いもの"""
    x = np.sort(np.ravel(x))
    k = int(np.floor(prob * len(x)))
    i = int(np.argmin(x[k:] - x[:len(x) - k]))
    return x[i], x[i + k]


def posterior_summary(idata, names):
    """平均・標準偏差・95%HDI・R-hat の表（変数がベクトルなら成分ごと）"""
    rows = []
    for nm in names:
        v = idata.posterior[nm].values                    # (チェーン, 点, …)
        v = v.reshape(v.shape[0], v.shape[1], -1)
        for k in range(v.shape[2]):
            x = v[:, :, k]
            lo, hi = hdi(x)
            rows.append({"name": nm if v.shape[2] == 1 else f"{nm}[{k}]", "mean": x.mean(),
                         "sd": x.std(ddof=1), "hdi_2.5%": lo, "hdi_97.5%": hi, "r_hat": rhat(x)})
    return pd.DataFrame(rows).set_index("name")


def pair_plot(idata, names, out_png):
    """パラメータどうしの散布図（ペアプロット）。対角は1変数のヒストグラム"""
    xs = [idata.posterior[nm].values.ravel() for nm in names]
    k = len(names)
    fig, ax = plt.subplots(k, k, figsize=(2.6 * k, 2.6 * k))
    for i in range(k):
        for j in range(k):
            if i == j:
                ax[i, j].hist(xs[i], bins=40, color="tab:blue", alpha=0.7)
            elif i > j:
                ax[i, j].plot(xs[j], xs[i], ".", ms=1, alpha=0.2)
            else:
                ax[i, j].axis("off")
            if i == k - 1:
                ax[i, j].set_xlabel(names[j])
            if j == 0 and i > 0:
                ax[i, j].set_ylabel(names[i])
    fig.tight_layout()
    fig.savefig(out_png, dpi=120)
    plt.close("all")


def posterior_band(idata, Vg, VT, ndraw=300):
    """平均曲線（オフセットなし）の 95% 信用帯"""
    post = idata.posterior
    L = post["log10_Is"].values.ravel()
    N = post["n"].values.ravel()
    LR = post["log_Rs"].values.ravel()
    idx = np.random.default_rng(0).choice(len(L), size=min(ndraw, len(L)), replace=False)
    curves = np.array([diode_current(Vg, L[i], N[i], LR[i], VT) for i in idx])
    lo, hi = np.percentile(curves, [2.5, 97.5], axis=0)
    return lo, hi


# ------------------------------------------------------------------ 図
def local_ideality(d, theta, VT):
    """L レンジ・往路の点で、7点ずつ ln I を V に直線回帰して局所的な n を求める"""
    s = d[(d["r"] == 0) & (d["dir"] == "up")].sort_values("V")
    V = s["V"].to_numpy()
    Ic = s["I"].to_numpy() - theta[3] * 1e-6
    ok = Ic > 5e-7
    lv, li = V[ok], np.log(Ic[ok])
    vv, nn = [], []
    for i in range(3, len(lv) - 3):
        b = np.polyfit(lv[i - 3:i + 4], li[i - 3:i + 4], 1)[0]
        if b > 0:
            vv.append(lv[i])
            nn.append(1.0 / (b * VT))
    return np.array(vv), np.array(nn)


def make_plots(d, VT, theta, band, Vg, out_png, title, meta):
    fig, ax = plt.subplots(2, 2, figsize=(11, 8))
    for rr, mk, lab in ((0, "o", "L レンジ"), (1, "s", "H レンジ")):
        s = d[d["r"] == rr]
        V = s["V"].to_numpy()
        Ic = s["I"].to_numpy() - theta[3 + rr] * 1e-6     # オフセット補正後
        ax[0, 0].plot(V, Ic * 1e3, mk, ms=3, alpha=0.7, label=lab)
        pos = Ic > 0
        ax[0, 1].semilogy(V[pos], Ic[pos], mk, ms=3, alpha=0.7, label=lab)
    Ifit = diode_current(Vg, theta[0], theta[1], theta[2], VT)
    ax[0, 0].plot(Vg, Ifit * 1e3, "k-", lw=1, label="最小二乗")
    ax[0, 1].semilogy(Vg, np.maximum(Ifit, 1e-12), "k-", lw=1, label="最小二乗")
    if band is not None:
        ax[0, 1].fill_between(Vg, np.maximum(band[0], 1e-12), np.maximum(band[1], 1e-12),
                              color="tab:red", alpha=0.3, label="事後95%信用帯")
    ax[0, 0].set(xlabel="V [V]", ylabel="I [mA]", title="I-V（線形）")
    ax[0, 1].set(xlabel="V [V]", ylabel="I [A]", title="I-V（片対数）", ylim=(1e-8, None))
    ax[0, 0].legend()
    ax[0, 1].legend()

    vv, nn = local_ideality(d, theta, VT)
    ax[1, 0].plot(vv, nn, ".")
    ax[1, 0].axhline(theta[1], color="k", lw=1)
    ax[1, 0].set(xlabel="V [V]", ylabel="局所的な理想係数 n", ylim=(0, 6), title="局所 n（L レンジ・往路）")

    s_all = sigma_total(d, meta)
    resid = (d["I"].to_numpy() - model_I(theta, d["V"].to_numpy(), d["r"].to_numpy(), VT)) / s_all
    ax[1, 1].plot(d["V"], resid, ".", ms=3)
    ax[1, 1].axhline(0, color="k", lw=1)
    ax[1, 1].set(xlabel="V [V]", ylabel="規格化残差", title="残差（模様があればモデルの不足）")
    fig.suptitle(title)
    fig.tight_layout()
    fig.savefig(out_png, dpi=150)
    print(f"図を保存しました: {out_png}")


# ------------------------------------------------------------------ メイン
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("csv")
    ap.add_argument("--no-bayes", action="store_true", help="最小二乗だけ行う")
    ap.add_argument("--draws", type=int, default=1000)
    ap.add_argument("--tune", type=int, default=1500)   # ノートブック版で追加（本文は 1500 に固定）
    a = ap.parse_args()

    check_lambertw()
    d, meta = load(a.csv)
    VT = KB * (meta["temp_C"] + 273.15) / Q
    print(f"DUT={meta['dut']}  T={meta['temp_C']:.1f}°C  VT={VT*1e3:.2f} mV  点数={len(d)}")

    theta0 = initial_guess(d, VT)
    theta, cov, chi2 = fit_lsq(d, VT, theta0, meta)
    se = np.sqrt(np.diag(cov))
    print("---- 重みつき最小二乗 ----")
    print(f"  n        = {theta[1]:.3f} ± {se[1]:.3f}")
    print(f"  log10 Is = {theta[0]:.2f} ± {se[0]:.2f}   （Is ≈ {10**theta[0]:.2e} A）")
    print(f"  Rs       ≈ {np.exp(theta[2]):.3g} Ω（ln Rs の標準誤差 {se[2]:.2f}）")
    print(f"  オフセット L = {theta[3]:+.3f} µA, H = {theta[4]:+.2f} µA,  規格化χ² = {chi2:.2f}")

    Vg = np.linspace(max(d["V"].min(), 0.0), d["V"].max(), 400)
    band = None
    csv_path = pathlib.Path(a.csv)
    if not a.no_bayes:
        print("---- PyMC（初回はコンパイルに時間がかかります） ----")
        idata = fit_bayes(d, VT, theta, a.draws, tune=a.tune)
        summ = posterior_summary(idata, ["n", "log10_Is", "Rs", "off_uA", "log_sigma_a", "sigma_rel"])
        print(summ.to_string(float_format=lambda v: f"{v:.4g}"))
        print(f"  発散（divergences）: {int(idata.sample_stats['diverging'].values.sum())}")
        summ.to_csv(csv_path.with_name(csv_path.stem + "_posterior.csv"))
        post = idata.posterior                          # 第6章で回路モデルに通すため、事後サンプルを組のまま保存
        np.savez(csv_path.with_name(csv_path.stem + "_samples.npz"), log10_Is=post["log10_Is"].values.ravel(),
                 n=post["n"].values.ravel(), Rs=post["Rs"].values.ravel(), temp_C=meta["temp_C"])
        band = posterior_band(idata, Vg, VT)
        pair_plot(idata, ["log10_Is", "n", "log_Rs"], csv_path.with_name(csv_path.stem + "_pair.png"))

    make_plots(d, VT, theta, band, Vg, csv_path.with_name(csv_path.stem + "_fit.png"),
               f"{meta['dut']}（{meta['temp_C']:.1f}°C）", meta)


if __name__ == "__main__":     # PyMC の並列サンプリングに必要
    main()

In [ ]:
%%writefile check_resistor.py
#!/usr/bin/env python3
"""第1章 手順6：抵抗を DUT にして測定系を確認する（傾き = 抵抗値）
使い方: uv run python check_resistor.py data_iv/R1k_20261121-100000.csv 998.7
        （最後の数値は DMM で測った抵抗値 [Ω]。省略可）
"""
import sys

import statsmodels.api as sm
from iv_analysis import load

df, meta = load(sys.argv[1])
R_dmm = float(sys.argv[2]) if len(sys.argv) > 2 else None
for rng, d in df.groupby("rng"):
    I = d["I"].to_numpy()
    V = d["V"].to_numpy()
    fit = sm.OLS(V, sm.add_constant(I)).fit()        # V = 切片 + R·I
    R = fit.params[1]
    lo, hi = fit.conf_int(alpha=0.05)[1]
    print(f"[{rng} レンジ] R = {R:.2f} Ω（95%CI {lo:.2f}〜{hi:.2f}）  切片 = {fit.params[0]*1e3:+.2f} mV"
          f"  残差SD = {fit.resid.std(ddof=2)*1e3:.2f} mV  最大電流 = {I.max()*1e3:.3f} mA")
    if R_dmm:
        print(f"           DMM との差: {(R / R_dmm - 1) * 100:+.2f} %")
print("※ V/I は ADC の比で決まるので VREF の誤差は打ち消し合う。ここで見ているのは R_L・R_H の値、")
print("  ADC の直線性、E9 の漏れ電流の有無。CI は統計誤差のみで、DMM の確度は含まない。")

In [ ]:
%%writefile planck.py
#!/usr/bin/env python3
"""LED 3色の I-V から一定電流での電圧 V_ref を求め、1/λ に回帰してプランク定数を推定する
使い方:
  uv run python planck.py data_iv/LED_IR_*.csv data_iv/LED_RED_*.csv data_iv/LED_GREEN_*.csv
"""
import sys

import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt
from iv_analysis import load       # 日本語フォント設定も読み込まれる

Q = 1.602176634e-19
C = 299792458.0
H_SI = 6.62607015e-34
IREFS = [1e-6, 3e-6, 1e-5, 3e-5]    # 基準電流 [A]（L レンジで測れる範囲）
V_ZERO = 0.5                        # V_A がこれより低い点は LED にほぼ電流が流れないとみなし、電流のゼロ点に使う [V]


def v_at_current(d, Iref):
    """往路の L レンジデータで、電流が Iref になる電圧を ln I の線形補間で求める"""
    d = d.sort_values("V")
    V = d["V"].to_numpy()
    I = np.maximum.accumulate(d["I"].to_numpy())     # ノイズで減る点をならして単調にする
    ok = I > 0
    if ok.sum() < 5 or Iref < I[ok].min() or Iref > I[ok].max():
        return np.nan
    return float(np.interp(np.log(Iref), np.log(I[ok]), V[ok]))


def main(paths):
    rows = []
    for p in paths:
        df, meta = load(p)
        lam = meta.get("wavelength_nm")
        if lam is None:
            print(f"{p}: wavelength_nm が無いのでスキップ")
            continue
        for rep, g in df[df["rng"] == "L"].groupby("rep"):
            # ADC のオフセット（V_K のゼロ点のずれ）を引く。1 µA では 0.5 mV のずれでも電流の 5% になる
            zero = g.loc[g["vA"] < V_ZERO, "I"]
            I0 = float(zero.median()) if len(zero) >= 3 else 0.0
            print(f"{meta['dut']} rep {rep}：電流のゼロ点 {I0*1e9:+.0f} nA（V_A < {V_ZERO} V の {len(zero)} 点の中央値）を引く")
            up = g[g["dir"] == "up"].assign(I=lambda x: x["I"] - I0)
            for Iref in IREFS:
                rows.append({"dut": meta["dut"], "lam_nm": lam, "rep": rep,
                             "Iref": Iref, "Vref": v_at_current(up, Iref)})
    t = pd.DataFrame(rows).dropna()
    if t.empty:
        sys.exit("有効なデータがありません")
    print(t.groupby(["dut", "Iref"])["Vref"].agg(["mean", "std", "count"]))
    print(f"\n参考：hc/q = {H_SI*C/Q*1e6:.4f} V·µm、h(SI) = {H_SI:.5e} J·s\n")

    fig, ax = plt.subplots(figsize=(6.5, 4.5))
    for Iref, g in t.groupby("Iref"):
        m = g.groupby(["dut", "lam_nm"], as_index=False)["Vref"].mean()   # LEDごとの平均
        if len(m) < 2:
            continue
        x = 1.0 / (m["lam_nm"].to_numpy() * 1e-3)       # 1/λ [1/µm]
        fit = sm.OLS(m["Vref"].to_numpy(), sm.add_constant(x)).fit()
        b = fit.params[1]                               # 傾き [V·µm]
        h = b * 1e-6 * Q / C
        line = f"I_ref = {Iref*1e6:4.0f} µA：傾き {b:.3f} V·µm → h = {h:.3e} J·s（SI値の {h/H_SI*100:.0f}%）"
        if len(m) >= 3:
            lo, hi = fit.conf_int(alpha=0.05)[1]
            line += f"  95%CI {lo*1e-6*Q/C:.2e}〜{hi*1e-6*Q/C:.2e}（自由度 {int(fit.df_resid)}）"
        print(line)
        # 悪い例：繰り返し測定を独立なデータとして扱うと、区間が見かけ上狭くなる（疑似反復）
        xa = 1.0 / (g["lam_nm"].to_numpy() * 1e-3)
        fa = sm.OLS(g["Vref"].to_numpy(), sm.add_constant(xa)).fit()
        if fa.df_resid > 0:
            lo2, hi2 = fa.conf_int(alpha=0.05)[1]
            print(f"      （疑似反復のまま計算した区間：{lo2*1e-6*Q/C:.2e}〜{hi2*1e-6*Q/C:.2e} ← 狭すぎる）")
        ax.plot(x, m["Vref"], "o", label=f"{Iref*1e6:g} µA")
        xx = np.linspace(x.min() * 0.95, x.max() * 1.05, 50)
        ax.plot(xx, fit.params[0] + b * xx, "-", lw=1)
    ax.set(xlabel="1/λ [1/µm]", ylabel="V_ref [V]", title="LED の立ち上がり電圧と 1/λ")
    ax.legend(title="基準電流")
    fig.tight_layout()
    fig.savefig("planck.png", dpi=150)
    print("\nplanck.png を保存しました")


if __name__ == "__main__":
    main(sys.argv[1:])

In [ ]:
%%writefile synth_iv.py
#!/usr/bin/env python3
"""第1章（ノートブック版で追加）：Pico がなくても解析を試すための合成の I-V データ
  iv_capture.py と同じ形の CSV（列 dut,rep,rng,dir,idx,duty,vA,vK,vS,sdA,sdK,sdS）と JSON を data_iv/ に作る。
  回路は図1-7：GP15 の PWM（D·V_OH）→ R_F → 節点A → DUT → 節点K → R_L → AGND
                （H レンジでは 節点K → 2N7000（オン抵抗 R_on）→ 節点S → R_H → AGND も並列に）
  定常状態（C_F の電流 0）で、PWM の平均 D·V_OH = I·R_F + V(I) + V_K(I) を I について解く。
  DUT が diode のときは 1-3-2 の式を V について解いた形 V(I) = n V_T ln(1 + I/I_s) + I R_s、
  resistor のときは V(I) = I R。各点の平均の読みには、1回の読みの雑音（標準偏差 --noise-mv）を
  nsamp 回平均した分の雑音と、ADC のオフセット（V_K に +0.5 mV、V_S に +0.3 mV）を足す。
  真の値（--log10-is・--n・--rs・--r）は解析する側（iv_analysis.py）には知らせない。DNL は入れていない。

  python synth_iv.py --dut 1N4148 --log10-is -8.602 --n 1.75 --rs 1.0 --temp 22.5
  python synth_iv.py --dut R1k --kind resistor --r 998.7 --npts 61
  python synth_iv.py --dut LED_RED --wavelength 625 --n 2.0 --v10 1.63 --reps 3 --npts 61
  （--v10 を与えると、10 µA での電圧が --v10 になるように I_s を決める）
"""
import argparse
import json
import pathlib

import numpy as np
from scipy.optimize import brentq

Q = 1.602176634e-19
KB = 1.380649e-23
TRUE_HW = dict(VOH=3.300, R_F=998.0, R_L=9980.0, R_H=99.7, R_ON=6.0, VREF=3.296,
               OFF_A=0.0, OFF_K=0.5e-3, OFF_S=0.3e-3)     # 回路の真の値（R_L・R_H は DMM で測った値として JSON にも書く）


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--dut", required=True)
    ap.add_argument("--kind", choices=["diode", "resistor"], default="diode")
    ap.add_argument("--log10-is", type=float, default=-8.602)   # 1N4148 の目安 I_s = 2.5e-9 A（図1-1）
    ap.add_argument("--n", type=float, default=1.75)
    ap.add_argument("--rs", type=float, default=1.0)
    ap.add_argument("--v10", type=float, default=None, help="10 µA での電圧 [V]（与えると I_s をこれから決める）")
    ap.add_argument("--r", type=float, default=998.7, help="resistor のときの抵抗値 [Ω]")
    ap.add_argument("--wavelength", type=float, default=None)
    ap.add_argument("--temp", type=float, default=22.5)
    ap.add_argument("--npts", type=int, default=121)
    ap.add_argument("--nsamp", type=int, default=64)
    ap.add_argument("--reps", type=int, default=1)
    ap.add_argument("--noise-mv", type=float, default=1.9, help="1回の読みの雑音の標準偏差 [mV]（1-3-3）")
    ap.add_argument("--seed", type=int, default=1)
    ap.add_argument("--outdir", default="data_iv")
    a = ap.parse_args()

    hw = TRUE_HW
    VT = KB * (a.temp + 273.15) / Q
    log10_is = a.log10_is
    if a.v10 is not None:
        log10_is = np.log10(10e-6 / np.expm1(a.v10 / (a.n * VT)))
    Is = 10.0 ** log10_is

    def v_dut(I):
        if a.kind == "resistor":
            return I * a.r
        return a.n * VT * np.log1p(I / Is) + I * a.rs

    rng_gen = np.random.default_rng(a.seed)
    sd1 = a.noise_mv * 1e-3
    sem = sd1 / np.sqrt(a.nsamp)
    up = [round(65535 * i / (a.npts - 1)) for i in range(a.npts)]
    duties = up + up[::-1][1:]                               # iv_tracer.py の updown と同じ並び
    rows = []
    for rep in range(a.reps):
        for rr in ("L", "H"):
            r_path = hw["R_ON"] + hw["R_H"]
            r_sense = 1.0 / (1.0 / hw["R_L"] + 1.0 / r_path) if rr == "H" else hw["R_L"]
            for idx, duty in enumerate(duties):
                vdac = duty / 65535 * hw["VOH"]
                if vdac <= 0:
                    I = 0.0
                else:
                    f = lambda I: I * (hw["R_F"] + r_sense) + v_dut(I) - vdac
                    I = brentq(f, 0.0, vdac / hw["R_F"], xtol=1e-16, rtol=1e-14)
                vK = I * r_sense
                vS = vK / r_path * hw["R_H"] if rr == "H" else 0.0
                vA = vK + v_dut(I)
                noisy = [v + off + rng_gen.normal(0.0, sem) for v, off in
                         ((vA, hw["OFF_A"]), (vK, hw["OFF_K"]), (vS, hw["OFF_S"]))]
                noisy = [min(max(v, 0.0), hw["VREF"]) for v in noisy]   # ADC は 0〜VREF の外を読めない
                sds = sd1 * (1.0 + 0.05 * rng_gen.standard_normal(3))
                rows.append({"dut": a.dut, "rep": rep, "rng": rr, "dir": "up" if idx < a.npts else "down",
                             "idx": idx, "duty": duty,
                             "vA": round(noisy[0], 5), "vK": round(noisy[1], 5), "vS": round(noisy[2], 5),
                             "sdA": round(sds[0], 5), "sdK": round(sds[1], 5), "sdS": round(sds[2], 5)})

    outdir = pathlib.Path(a.outdir)
    outdir.mkdir(parents=True, exist_ok=True)
    base = outdir / f"{a.dut}_synth"
    keys = list(rows[0].keys())
    with open(base.with_suffix(".csv"), "w") as fp:
        fp.write(",".join(keys) + "\n")
        for r in rows:
            fp.write(",".join(str(r[k]) for k in keys) + "\n")
    meta = {"dut": a.dut, "wavelength_nm": a.wavelength, "temp_C": a.temp, "vref": hw["VREF"],
            "R_L": hw["R_L"], "R_H": hw["R_H"], "R_F": 1000.0, "npts": a.npts, "settle_ms": 600,
            "nsamp": a.nsamp, "reps": a.reps, "datetime": "synth"}
    base.with_suffix(".json").write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8")
    truth = {"kind": a.kind, "temp_C": a.temp,      # 電流のオフセット：L レンジは V_K の分だけ、H レンジは V_S の分も
             "off_uA": [hw["OFF_K"] / hw["R_L"] * 1e6, (hw["OFF_K"] / hw["R_L"] + hw["OFF_S"] / hw["R_H"]) * 1e6]}
    truth.update({"R": a.r} if a.kind == "resistor" else {"log10_Is": float(log10_is), "n": a.n, "Rs": a.rs})
    base.with_name(base.name + "_truth.json").write_text(json.dumps(truth, indent=2), encoding="utf-8")
    print(f"保存しました: {base}.csv / .json（{len(rows)} 点）")
    rnd = lambda v: round(v, 4) if isinstance(v, float) else [round(x, 4) for x in v] if isinstance(v, list) else v
    print("真の値（_truth.json。解析では使わない）:", {k: rnd(v) for k, v in truth.items()})


if __name__ == "__main__":
    main()

In [ ]:
import glob, json, os, re, shlex, subprocess, sys
import numpy as np
import pandas as pd
from IPython.display import Image, display


def run(args):
    """ターミナルの「uv run python ○○.py …」の代わり。表示を出し、あとで確かめられるように文字列で返す"""
    words = []
    for w in shlex.split(args):   # data_iv/LED_*.csv のような * を、ターミナルと同じく名前の順に広げる
        words += (sorted(glob.glob(w)) or [w]) if any(c in w for c in "*?[") else [w]
    p = subprocess.run([sys.executable] + words, capture_output=True, text=True)
    print(p.stdout, end="")
    keep = [l for l in p.stderr.splitlines()
            if "findfont" not in l and not re.search(r"[━─]|Progress|Draw|Step size|Grad evals|^\s*$", l)]   # 候補のうち見つからないフォントの警告と、PyMC の進み具合の表は出さない（日本語のフォントがあるかは、0-1 のセルの表示で分かる）
    if keep:
        print("\n".join(keep), file=sys.stderr)
    assert p.returncode == 0, "★実行に失敗：" + args
    return p.stdout


def show(*paths, width=800):
    """保存された図を表示する"""
    for f in paths:
        print(f)
        display(Image(filename=f, width=width))


def ok(cond, msg):
    assert cond, "★確かめ失敗：" + msg
    print("OK", msg)


def num(pattern, text, i=1):
    """表示（text）から、pattern の (…) の数を取り出す"""
    m = re.search(pattern, text)
    assert m, "表示に見つからない：" + pattern
    return float(m.group(i).replace("−", "-"))


def truth(dut):
    """synth_iv.py が書いた真の値（確かめだけに使う）"""
    return json.loads(open(f"data_iv/{dut}_synth_truth.json", encoding="utf-8").read())


def lsq(text):
    """iv_analysis.py の「重みつき最小二乗」の表示を数にする"""
    return dict(n=num(r"n        = ([\d.]+) ±", text), se_n=num(r"n        = [\d.]+ ± ([\d.]+)", text),
                log10_Is=num(r"log10 Is = (-?[\d.]+) ±", text), se_log10_Is=num(r"log10 Is = -?[\d.]+ ± ([\d.]+)", text),
                Rs=num(r"Rs       ≈ ([\d.e+-]+) Ω", text), se_lnRs=num(r"ln Rs の標準誤差 ([\d.]+)", text),
                chi2=num(r"規格化χ² = ([\d.]+)", text))


print("準備ができました")

## 1. 合成の I-V データを作る（Pico A の測定の代わり）

`synth_iv.py` で、次の4種類を `data_iv/` に作ります（数秒）。回路の値は、R_L = 9980 Ω、R_H = 99.7 Ω（DMM で測った値として JSON にも書く）、R_F = 998 Ω、2N7000 のオン抵抗 6 Ω、ADC_VREF = 3.296 V、室温 22.5°C です。ADC のオフセットは V_K に +0.5 mV、V_S に +0.3 mV としました。電流に直すと、L レンジ（1-3-3 の式で $V_K/R_L$ だけ）では約 0.05 µA、H レンジ（$V_S/R_H$ も足す）では約 3.06 µA のずれになります（`_truth.json` の `off_uA` は [L, H] の順）。

| DUT | 真の値 | 掃引（本文のオプションで言うと） | 使う節 |
|---|---|---|---|
| `R1k` | 998.7 Ω の抵抗 | `--npts 61` | 2 |
| `1N4148` | $n = 1.75$、$I_s = 2.5\times10^{-9}$ A、$R_s = 1$ Ω（図1-1 と同じ目安） | `--npts 121 --reps 1` | 3・4・5 |
| `1N4148_R100` | 同じ 1N4148 に 100 Ω を直列につないだもの（$R_s = 101$ Ω） | `--npts 121 --reps 1` | 5 |
| `LED_IR`・`LED_RED`・`LED_GREEN` | 10 µA での電圧が 0.97・1.63・2.26 V（緑は余分な電圧つき。$n$ は 2・2・3） | `--npts 61 --reps 3` | 6 |

LED の 10 µA での電圧は、1-3-5 の $V_{\mathrm{ref}} = \dfrac{hc}{q}\cdot\dfrac{1}{\lambda} - \dfrac{\delta}{q}$ で $\delta/q = 0.35$ V としたもの（赤外 0.97 V、赤 1.63 V）に、緑だけ 0.25 V を足した値です（緑の InGaN 系で $\delta$ が一定という仮定が崩れることのまね）。

In [ ]:
run("synth_iv.py --dut R1k --kind resistor --r 998.7 --npts 61")
run("synth_iv.py --dut 1N4148 --log10-is -8.602 --n 1.75 --rs 1.0 --temp 22.5")
run("synth_iv.py --dut 1N4148_R100 --log10-is -8.602 --n 1.75 --rs 101.0 --temp 22.5")
run("synth_iv.py --dut LED_IR --wavelength 940 --n 2.0 --v10 0.97 --reps 3 --npts 61 --seed 3")
run("synth_iv.py --dut LED_RED --wavelength 625 --n 2.0 --v10 1.63 --reps 3 --npts 61 --seed 4")
out_syn = run("synth_iv.py --dut LED_GREEN --wavelength 525 --n 3.0 --v10 2.26 --reps 3 --npts 61 --seed 5")

In [ ]:
# 確かめ：iv_capture.py と同じ形のファイルができた
d = pd.read_csv("data_iv/1N4148_synth.csv")
meta = json.load(open("data_iv/1N4148_synth.json", encoding="utf-8"))
ok(list(d.columns) == ["dut", "rep", "rng", "dir", "idx", "duty", "vA", "vK", "vS", "sdA", "sdK", "sdS"],
   "CSV の列が iv_capture.py と同じ")
ok({"dut", "temp_C", "vref", "R_L", "R_H", "nsamp", "wavelength_nm"} <= set(meta), "JSON に解析で使う条件がそろっている")
ok(len(d) == 2 * (2 * 121 - 1) and set(d["rng"]) == {"L", "H"} and set(d["dir"]) == {"up", "down"},
   "1N4148：L・H レンジの往復（折り返し点は1回）で 482 点")
I_H = (d["vK"] / meta["R_L"] + d["vS"] / meta["R_H"])[d["rng"] == "H"].max()
ok(2.0e-3 < I_H < 2.6e-3, f"H レンジの最大電流 {I_H*1e3:.2f} mA（1-3-3 の見積もり 約 2.4 mA）")

## 2. 抵抗の校正（手順 6）

教科書では、校正用の 1 kΩ を DUT の位置に挿して測り、DMM で測った値（ここでは 998.7 Ω）と比べます。傾き $V/I$ が抵抗値です。

In [ ]:
out_r = run("check_resistor.py data_iv/R1k_synth.csv 998.7")

In [ ]:
# 確かめ（手順 6：L レンジの推定値が DMM の値と ±1% 程度以内で一致する）
RL_est = num(r"\[L レンジ\] R = ([\d.]+) Ω", out_r)
lo, hi = num(r"\[L レンジ\] R = [\d.]+ Ω（95%CI ([\d.]+)〜", out_r), num(r"\[L レンジ\] R = [\d.]+ Ω（95%CI [\d.]+〜([\d.]+)）", out_r)
ok(abs(RL_est / 998.7 - 1) < 0.01, f"L レンジ R = {RL_est} Ω が DMM の値 998.7 Ω と ±1% 以内")
ok(lo < 998.7 < hi, f"L レンジの 95%CI（{lo}〜{hi} Ω）に真の値 998.7 Ω が入る")
ok("[H レンジ] R =" in out_r, "H レンジの推定値も表示された（本文：H レンジは ±数% ずれることがある）")

## 3. 重みつき最小二乗（手順 10 の `--no-bayes`）

`--no-bayes` を付けると、最小二乗だけを行います（数秒）。最初の行の「自己診断」は、ニュートン法で解いた自作の Lambert W（1-3-2 の $w \leftarrow w\,(1 + x - \ln w)/(1 + w)$ を8回）と scipy の値の比較です。

In [ ]:
out_lsq = run("iv_analysis.py data_iv/1N4148_synth.csv --no-bayes")
show("data_iv/1N4148_synth_fit.png")

In [ ]:
# 確かめ（1-3-2 と手順 10）
t = truth("1N4148")
f = lsq(out_lsq)
ok(num(r"最大相対誤差: ([\d.e+-]+)", out_lsq) < 1e-10, "自作 Lambert W が scipy と 1e-10 以内で一致")
ok(abs(f["n"] - t["n"]) < 4 * f["se_n"] + 0.01, f"n = {f['n']} が真の値 {t['n']} に近い（標準誤差 {f['se_n']} の4倍以内）")
ok(abs(f["log10_Is"] - t["log10_Is"]) < 4 * f["se_log10_Is"] + 0.02,
   f"log10 Is = {f['log10_Is']} が真の値 {t['log10_Is']:.3f} に近い")
ok(0.2 < f["chi2"] < 3, f"規格化χ² = {f['chi2']}（1 に近い。雑音の見積もり sigma_total が大きくは外れていない）")
ok(os.path.exists("data_iv/1N4148_synth_fit.png") and not os.path.exists("data_iv/1N4148_synth_samples.npz"),
   "_fit.png だけが保存され、--no-bayes では事後サンプルは作られない")

## 4. PyMC でのベイズ推定と、事後サンプルの保存（手順 10、1-3-4）

同じ CSV を、今度は `--no-bayes` なしで解析します。最小二乗の結果を初期値にして NUTS（4 チェーン）でサンプリングし、平均・sd・95%HDI・R-hat の表、発散の数を表示して、`_posterior.csv`（表）、`_samples.npz`（第6章で使う事後サンプル）、`_pair.png`（ペアプロット）、`_fit.png`（事後 95% 信用帯つき）を保存します。初回は PyTensor のコンパイルに時間がかかります（執筆時の Mac で全部で約1分）。

**ノートブックでは小さい設定**（`DRAWS = 300`、`TUNE = 300`）を既定にしています。教科書の本番にするときは、次のセルを `DRAWS = 1000`、`TUNE = 1500` に書きかえてください（`uv run python iv_analysis.py data_iv/….csv` の既定と同じ。執筆時の Mac で数分）。

In [ ]:
DRAWS = 300   # 本番（教科書の既定）：DRAWS = 1000
TUNE = 300    # 本番（教科書の既定）：TUNE = 1500
out_bayes = run(f"iv_analysis.py data_iv/1N4148_synth.csv --draws {DRAWS} --tune {TUNE}")
show("data_iv/1N4148_synth_pair.png", width=600)
show("data_iv/1N4148_synth_fit.png")

In [ ]:
# 確かめ（手順 10 と 1-3-4：真の値が 95%HDI に入る、R-hat、発散）
t = truth("1N4148")
post = pd.read_csv("data_iv/1N4148_synth_posterior.csv", index_col="name")
for nm in ["n", "log10_Is", "Rs"]:
    lo, hi = post.loc[nm, "hdi_2.5%"], post.loc[nm, "hdi_97.5%"]
    ok(lo <= t[nm] <= hi, f"{nm} の真の値 {t[nm]:.4g} が 95%HDI [{lo:.4g}, {hi:.4g}] に入る")
for k in (0, 1):
    lo, hi = post.loc[f"off_uA[{k}]", "hdi_2.5%"], post.loc[f"off_uA[{k}]", "hdi_97.5%"]
    ok(lo <= t["off_uA"][k] <= hi, f"オフセット（{'LH'[k]} レンジ）の真の値 {t['off_uA'][k]:.3g} µA が 95%HDI [{lo:.3g}, {hi:.3g}] に入る")
rh = post["r_hat"]
ok(rh.max() < 1.05, f"R-hat の最大 {rh.max():.3f} が 1.05 未満（本文の目安 1.01 未満：{'満たす' if rh.max() < 1.01 else '小さい設定では超えた。本番の設定で確かめる'}）")
div = int(num(r"発散（divergences）: (\d+)", out_bayes))
ok(div <= 0.01 * 4 * DRAWS, f"発散 {div} 回（0 か少数）")

In [ ]:
# 確かめ（図1-5 の ⑦：_samples.npz の中身。第6章の clipper_model.load_posterior が読む形）
z = np.load("data_iv/1N4148_synth_samples.npz")
ok(sorted(z.files) == ["Rs", "log10_Is", "n", "temp_C"], f"_samples.npz の中身は log10_Is・n・Rs・temp_C（δ・σ は入れない）：{sorted(z.files)}")
ok(all(z[k].shape == (4 * DRAWS,) for k in ["log10_Is", "n", "Rs"]), f"log10_Is・n・Rs は 4 チェーン × {DRAWS} = {4 * DRAWS} 組")
ok(float(z["temp_C"]) == json.load(open("data_iv/1N4148_synth.json"))["temp_C"], "temp_C は JSON の室温（22.5°C）")
ok((z["Rs"] > 0).all() and (0.8 <= z["n"]).all() and (z["n"] <= 6).all(), "Rs > 0、n は事前分布の範囲 [0.8, 6] の中")
ok(abs(z["n"].mean() - post.loc["n", "mean"]) < 1e-3 and abs(z["log10_Is"].mean() - post.loc["log10_Is", "mean"]) < 1e-3,
   "_samples.npz の平均が _posterior.csv の表と一致する（同じサンプル）")
rho = np.corrcoef(z["log10_Is"], z["n"])[0, 1]
ok(rho > 0.9, f"log10 Is と n の相関 {rho:.3f}：強い正の相関（図1-4 (b)。だから組のまま保存する）")
rho_shuf = np.corrcoef(z["log10_Is"], np.random.default_rng(0).permutation(z["n"]))[0, 1]
print(f"（参考：n だけを並べ替えると相関は {rho_shuf:+.3f} になり、組の情報がなくなる。第6章の (誤) の区間はこれ）")

## 5. 識別可能性：$R_s$ が決まるとき・決まらないとき（1-3-3・1-3-4 のコツ、1-7）

4 節の合成データには DNL のような系統誤差が入っていないので、H レンジの 1 mA 以上の点（$IR_s \approx 1$〜2.4 mV）から、$R_s = 1$ Ω が1Ω前後に絞られました（1-7 の「H レンジの1mA 以上の点が多く雑音が小さいときは1Ω前後に絞られることもある」の場合）。ここでは、条件を変えた2つで比べます。

- (a) **L レンジだけ**のデータ（最大約 0.25 mA。$IR_s$ は約 0.25 mV で、64 回平均の雑音と同じくらい）：$R_s$ は、データからは「数 Ω より大きくはない」ことしか言えず、区間は下側（小さい $R_s$）に何桁も広がるはずです（L レンジの電流の雑音は約 23 nA と小さいので、$R_s$ が数 Ω 以上あると曲がりが見えてしまい、上側だけは絞られます。事前分布は $\ln R_s \sim \mathcal N(\ln 5, 2^2)$）。最小二乗の標準誤差が、この片側だけ決まる様子を表せるかも見ます。4 節の CSV から H レンジの行を除いたものを作って、同じ設定でベイズ推定します。
- (b) **100 Ω を直列につないだ 1N4148**（$R_s = 101$ Ω）：図1-4 (a) のとおり数 µA から $IR_s$ が雑音を超えるので、最小二乗だけで $R_s$ がはっきり決まるはずです。

In [ ]:
# (a) H レンジの行を除いた CSV と、同じ条件の JSON を作る
import shutil
d = pd.read_csv("data_iv/1N4148_synth.csv")
d[d["rng"] == "L"].to_csv("data_iv/1N4148_Lonly_synth.csv", index=False)
shutil.copy("data_iv/1N4148_synth.json", "data_iv/1N4148_Lonly_synth.json")
shutil.copy("data_iv/1N4148_synth_truth.json", "data_iv/1N4148_Lonly_synth_truth.json")
out_L = run(f"iv_analysis.py data_iv/1N4148_Lonly_synth.csv --draws {DRAWS} --tune {TUNE}")
show("data_iv/1N4148_Lonly_synth_pair.png", width=600)

In [ ]:
# (b) 100 Ω を直列につないだもの（最小二乗だけ）
out_R100 = run("iv_analysis.py data_iv/1N4148_R100_synth.csv --no-bayes")

In [ ]:
# 確かめ（1-3-4 のコツ：R_s が効くのは IR_s が雑音より大きいときだけ）
pA = pd.read_csv("data_iv/1N4148_synth_posterior.csv", index_col="name")
pL = pd.read_csv("data_iv/1N4148_Lonly_synth_posterior.csv", index_col="name")
wA = pA.loc["Rs", "hdi_97.5%"] / pA.loc["Rs", "hdi_2.5%"]
wL = pL.loc["Rs", "hdi_97.5%"] / max(pL.loc["Rs", "hdi_2.5%"], 1e-300)
ok(wL > 10 and wL > 5 * wA,
   f"L レンジだけでは R_s の 95%HDI が {pL.loc['Rs', 'hdi_2.5%']:.3g}〜{pL.loc['Rs', 'hdi_97.5%']:.3g} Ω（上下の比 {wL:.3g} 倍）と何桁にも広がる"
   f"（L・H 両方では {pA.loc['Rs', 'hdi_2.5%']:.3g}〜{pA.loc['Rs', 'hdi_97.5%']:.3g} Ω、比 {wA:.3g} 倍）")
ok(pL.loc["Rs", "hdi_2.5%"] <= 1.0 <= pL.loc["Rs", "hdi_97.5%"], "それでも真の値 1 Ω は区間に入る（失敗ではなく「データが何も言っていない」）")
zL = np.load("data_iv/1N4148_Lonly_synth_samples.npz")
sd_lnRs = np.log(zL["Rs"]).std()
ok(sd_lnRs > 0.5, f"L レンジだけの ln R_s の事後 sd {sd_lnRs:.2f}（事前分布の sd は 2。L・H 両方では {np.log(np.load('data_iv/1N4148_synth_samples.npz')['Rs']).std():.2f}）")
ok(pL.loc["n", "hdi_2.5%"] <= 1.75 <= pL.loc["n", "hdi_97.5%"], "L レンジだけでも n は決まり、真の値 1.75 が区間に入る")
ok(np.corrcoef(zL["log10_Is"], zL["n"])[0, 1] > 0.9, "L レンジだけでも log10 Is と n は強い正の相関")
fL = lsq(out_L)
lo_l, hi_l = fL["Rs"] * np.exp(-1.96 * fL["se_lnRs"]), fL["Rs"] * np.exp(1.96 * fL["se_lnRs"])
ok(hi_l / lo_l < wL / 10,
   f"最小二乗は R_s ≈ {fL['Rs']:.3g} Ω（ln Rs の標準誤差 {fL['se_lnRs']:.2g}、±1.96 倍で {lo_l:.2g}〜{hi_l:.2g} Ω、比 {hi_l / lo_l:.2g} 倍）と、"
   f"ベイズの区間（比 {wL:.3g} 倍）よりずっと狭く見える（最適値のまわりの2次の近似なので、下側に広がる事後分布を表せない）")
fR = lsq(out_R100)
ok(abs(fR["Rs"] / 101 - 1) < 0.05 and fR["se_lnRs"] < 0.05,
   f"100 Ω を直列にすると R_s = {fR['Rs']:.4g} Ω（真の値 101 Ω）と、最小二乗だけではっきり決まる（ln Rs の標準誤差 {fR['se_lnRs']}）")

## 6. LED 3色からプランク定数（手順 11、1-3-5）

`planck.py` は、各 LED の L レンジ・往路のデータで、基準電流 $I_{\mathrm{ref}}$（1・3・10・30 µA）が流れる電圧 $V_{\mathrm{ref}}$ を求め、$1/\lambda$ に回帰して傾きから $h$ を求めます。3色そろえば自由度1の 95% 区間と、くり返し測定を独立なデータとして扱った（疑似反復の）狭すぎる区間も表示します。

L レンジの電流は $V_K/R_L$ だけから求めるので（1-3-3）、雑音は約 23 nA です。ただし V_K のオフセット（この合成データでは +0.5 mV → 約 50 nA）は 1 µA の 5% になるので、`planck.py` は $V_A < 0.5$ V の点の電流の中央値を「電流のゼロ点」として、くり返しごとに引いてから $V_{\mathrm{ref}}$ を求めます（1-3-5）。最初に表示されるゼロ点が、真の値の約 50 nA に近いことも確かめます。

In [ ]:
out_h = run("planck.py data_iv/LED_IR_*.csv data_iv/LED_RED_*.csv data_iv/LED_GREEN_*.csv")
show("planck.png", width=600)

In [ ]:
# 確かめ（手順 11 と 1-7：緑の余分な電圧のために h が大きめに出る、自由度 1 で区間が非常に広い）
H_SI = 6.62607015e-34
t_ir = truth("LED_IR")
I0s = [float(x) for x in re.findall(r"電流のゼロ点 ([+-]\d+) nA", out_h)]
ok(len(I0s) == 9 and all(abs(x - t_ir["off_uA"][0] * 1e3) < 25 for x in I0s),
   f"電流のゼロ点 {I0s} nA が、3色×3回とも真の値 {t_ir['off_uA'][0] * 1e3:.0f} nA から ±25 nA 以内")
for Iref in (1, 3, 10, 30):
    h = num(rf"I_ref = +{Iref} µA：傾き [\d.]+ V·µm → h = ([\d.e+-]+) J·s", out_h)
    lo = num(rf"I_ref = +{Iref} µA：.*?95%CI ([-\d.e+]+)〜", out_h)
    hi = num(rf"I_ref = +{Iref} µA：.*?95%CI [-\d.e+]+〜([-\d.e+]+)（自由度 1）", out_h)
    plo = num(rf"I_ref = +{Iref} µA：[^\n]*\n +（疑似反復のまま計算した区間：([-\d.e+]+)〜", out_h)
    phi = num(rf"I_ref = +{Iref} µA：[^\n]*\n +（疑似反復のまま計算した区間：[-\d.e+]+〜([-\d.e+]+)", out_h)
    ok(1.0 < h / H_SI < 1.5, f"I_ref = {Iref} µA：h = {h:.3e} J·s は SI 値の {h / H_SI * 100:.0f}%（1〜1.5 倍。緑の余分な電圧のため大きめ）")
    ok(lo < H_SI < hi and (hi - lo) > 5 * (phi - plo),
       f"I_ref = {Iref} µA：自由度 1 の 95%CI（{lo:.2e}〜{hi:.2e}）は SI 値を含み、疑似反復の区間（{plo:.2e}〜{phi:.2e}）より何倍も広い")

## おわりに

- 答えのわかった合成の I-V データで、`iv_analysis.py` の最小二乗とベイズ推定が真の値（$n$、$I_s$、オフセット）を区間に含むこと、R-hat と発散の確かめ方、第6章で使う `_samples.npz` の中身（log10_Is・n・Rs の組と室温）を確かめました。
- $\log_{10}I_s$ と $n$ は、どのデータでも強い正の相関を持ちます（図1-4 (b)）。だから事後サンプルは組のまま保存し、第6章で組のまま回路モデルに通します。
- $R_s$ は、$IR_s$ が雑音より大きくなる点があるときだけ決まります。L レンジだけ（$IR_s \approx 0.25$ mV）では上側しか決まらず、下側に何桁も広がりました（最小二乗の標準誤差は、それをずっと狭い区間に見せてしまいます）。100 Ω を直列にすると最小二乗だけでも決まりました。実物のデータには DNL などの系統誤差もあるので、1N4148 の $R_s$ は 4 節の合成データより広くなることが多いはずです（1-7）。
- プランク定数は、緑の余分な電圧のために大きめに出ること、自由度 1 の区間が非常に広いこと、疑似反復で区間が見かけ上狭くなることを見ました。
- 4・5 節を教科書の本番（`DRAWS = 1000`、`TUNE = 1500`）にすると、R-hat が本文の目安の 1.01 未満になるかも確かめられます。
- このノートブックに入れていないもの：回路を組むこと、Pico A での測定（手順 1〜9 の実物の部分）。これらは Mac で教科書の手順のとおりに行います。